# Model selection — WDR91 DEL → E-ASMS

Train a ranker on the DEL training data (`crosstalk_train`, labelled) and rank the E-ASMS test set (`crosstalk_test_inputs`, unlabelled).

Training data is from DEL (DNA-encoded library screening); the test is from E-ASMS (a different assay) — so this is a cross-method generalization task. All model decisions will be validated on DEL data only (building-block-grouped cross-validation); the E-ASMS test is untouched. 

Then predications will be submitted and scored on the [Kaggle competition leaderboard](https://www.kaggle.com/competitions/cross-talk-round-4/overview), where scores are based on **Hits@200** — amount of true binders in the top 200 ranked compounds.

> **Pipeline:**  
> Grouped CV → model selection (5 models) → fingerprint sweep + decorrelation → per-fingerprint tuning → ensemble comparison → domain-shift diagnosis → confidence weighting → final baseline submission files

---

## 1. Setup

Fingerprints are stored as comma-separated **count** strings, so parse them to vectors (counts feed the trees; binarize only later for similarity).  

I also parse the building-block IDs out of `DEL_ID` (format `L<lib>-<BB1>-<BB2>-<BB3>`) in this section — these group the cross-validation folds so combinatorial siblings don't leak across train/validation.  

Metric helpers (AUPRC, Hits@k) are defined here so every choice downstream is scored the same way. 

### <u>Data loading</u>

In [1]:
import numpy as np, pandas as pd
from scipy.sparse import csr_matrix
from sklearn.model_selection import (train_test_split, StratifiedGroupKFold,
                                     cross_validate, cross_val_predict, RandomizedSearchCV)
from sklearn.metrics import average_precision_score, roc_auc_score

TRAIN_PATH = "../input_data/raw/crosstalk_train.parquet"
TEST_PATH  = "../input_data/raw/crosstalk_test_inputs.parquet"

def parse_fp(s): return np.array(s.split(","), dtype=np.float32)
def fp_matrix(df, fp): return csr_matrix(np.stack(df[fp].map(parse_fp).to_numpy()))

train = pd.read_parquet(TRAIN_PATH)
y_train = train["DELLabel"].to_numpy()

# parse building blocks out of DEL_ID  (L<lib>-<BB1>-<BB2>-<BB3>) for grouped CV
parts = train["DEL_ID"].str.split("-", expand=True)
train["LIB"], train["BB1"], train["BB2"], train["BB3"] = parts[0], parts[1], parts[2], parts[3]

print("train:", train.shape, "| hit rate:", round(y_train.mean(), 4))
print("unique BB1/BB2/BB3:", train["BB1"].nunique(), train["BB2"].nunique(), train["BB3"].nunique())

train: (375595, 20) | hit rate: 0.0766
unique BB1/BB2/BB3: 2485 2319 3561


**Loaded:** 375,595 training compounds, hit rate 7.66%

**Building blocks parsed from `DEL_ID`:** 
- BB1 = 2,485 unique
- BB2 = 2,319 unique
- BB3 = 3,561 unique
- BB2 has the fewest → group cross-validation folds on BB2 (fewest, largest groups = most conservative split, keeps combinatorial siblings together)

### <u>The evaluation metrics are:</u>
1) **AUPRC** (precision-recall area, standard for imbalanced ranking as we have low hit rates) 
2) **Hits@k** (true positives in the top k — directly aligned with the Kaggle scoring)

Accuracy is avoided as an evaluation metric, meaningless at a ~8% hit rate.

In [3]:
def precision_at_k(y, s, k):
    k = min(k, len(y)); return y[np.argsort(s)[::-1][:k]].sum() / k
def hits_at_k(y, s, k=200):
    k = min(k, len(y)); return int(y[np.argsort(s)[::-1][:k]].sum())
def evaluate(y, s, name, ks=(50, 100, 200)):
    print(f"\n{name}  (n={len(y)}, hits={int(y.sum())}, rate={y.mean():.4f})")
    print(f"  AUPRC : {average_precision_score(y, s):.4f}  (random={y.mean():.4f})")
    print(f"  AUROC : {roc_auc_score(y, s):.4f}")
    for k in ks: print(f"  Hits@{k}: {hits_at_k(y, s, k)} / {k}")

---

## 2. Cross-validation: grouped on building blocks (BB2)

DEL compounds are combinatorial (BB1+BB2+BB3), so a random split could let near-sibling molecules (sharing building blocks) fall on both sides. This could mean the model half-memorizes and thus validation scores become inflated. 

Thus, I grouped folds by _BB2_ (fewest/largest group = most conservative) and used _StratifiedGroupKFold_ to keep the ~7.7% hit ratio constant across folds. All model and fingerprint choices are scored on this grouped CV.

A 60,000-compound subsample (`sub_idx`) for model/fingerprint selection is used instead of all 375k samples. Using the full dataset with the various models and folds would be too slow. A 60k stratified subsample is big enough to rank models reliably but fast enough to iterate.


In [2]:
# stratified 60k subsample for model/fingerprint selection
sub_idx, _ = train_test_split(np.arange(len(y_train)), train_size=60000,
                              stratify=y_train, random_state=42)
y_sub = y_train[sub_idx]

GROUP_COL = "BB2"
groups_sub = train[GROUP_COL].to_numpy()[sub_idx]
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

print("subsample:", len(sub_idx), "| hit rate:", round(y_sub.mean(), 4))
print("grouping on:", GROUP_COL, "| groups in subsample:", len(np.unique(groups_sub)))

subsample: 60000 | hit rate: 0.0766
grouping on: BB2 | groups in subsample: 1507


### <u>Output:</u>
60,000-compound stratified subsample (hit rate 0.0766, matching the full set), grouped on BB2.
- Results in → 1,507 groups, ~300 per fold
- Enough groups for clean 5-fold splits
- All selection below runs on this grouped CV

---
## 3. Model selection

The initial step is to compare five tree ensembles — RandomForest, ExtraTrees, XGBoost, CatBoost, LightGBM — on the same dataset (ECFP4, grouped CV), scored on AUPRC. This is just to get an idea of which models are worth building on.

> All five models are compared on a single fixed fingerprint — ECFP4, the field-standard Morgan default — so that differences reflect the model, not the representation

- **RandomForest (RF):** builds many decision trees independently on random data subsets, then averages them ("bagging"). Robust, stable, and gives a natural uncertainty estimate (tree disagreement).
- **ExtraTrees:** like RandomForest, but chooses split thresholds *randomly* rather than searching for the best ("extremely randomized trees"). Even more decorrelated than RF, though each individual tree is slightly weaker — another bagging method.
- **XGBoost:** builds trees sequentially, each correcting the previous one's errors ("boosting"). Often more accurate and more tunable.
- **CatBoost:** another boosting variant, with ordered boosting and symmetric trees designed to reduce overfitting.
- **LightGBM:** also boosting, but grows trees *leaf-wise* (splitting wherever the gain is highest) rather than level-by-level, and handles sparse, high-cardinality features efficiently. Often a strong performer on fingerprint data.

These span both major tree-ensemble families — **bagging** (RF, ExtraTrees) and **boosting** (XGBoost, CatBoost, LightGBM) — so this intial analysis covers different modelling strategies.

Each handles the class imbalance through its own weighting (`class_weight` / `scale_pos_weight` / `auto_class_weights` / `is_unbalance`). The goal is to pick the strongest learners to carry forward and any clearly weak models are dropped before the more time-expensive ensemble and tuning steps.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")  # for LightGBM warnings

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

X_ecfp = fp_matrix(train, "ECFP4")
X_sub = X_ecfp[sub_idx]
spw = (y_sub == 0).sum() / (y_sub == 1).sum()

models = {
    "RF":          RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                          n_jobs=-1, random_state=42),
    "ExtraTrees":  ExtraTreesClassifier(n_estimators=200, class_weight="balanced",
                                        n_jobs=-1, random_state=42),
    "XGBoost":     XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                 scale_pos_weight=spw, tree_method="hist",
                                 n_jobs=-1, random_state=42, eval_metric="aucpr"),
    "CatBoost":    CatBoostClassifier(iterations=300, depth=6, learning_rate=0.1,
                                      auto_class_weights="Balanced", verbose=0, random_seed=42),
    "LightGBM":    LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                                  is_unbalance=True, n_jobs=-1, random_state=42, verbose=-1),
}

print(f"{'model':12s} AUPRC (5 grouped folds)")
for name, m in models.items():
    r = cross_validate(m, X_sub, y_sub, groups=groups_sub, cv=sgkf,
                       scoring="average_precision", n_jobs=-1)
    print(f"{name:12s} {r['test_score'].mean():.4f} ± {r['test_score'].std():.4f}")

model        AUPRC (5 grouped folds)


python(24447) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(24448) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(24449) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(24450) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(24451) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(24452) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(24453) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(24454) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(24455) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(24456) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(24457) Malloc

RF           0.5227 ± 0.0443
ExtraTrees   0.5201 ± 0.0467
XGBoost      0.4967 ± 0.0486
CatBoost     0.4770 ± 0.0442
LightGBM     0.5318 ± 0.0520


### <u>Initial Model Result (grouped CV on ECFP4, AUPRC):</u>

| Model | AUPRC | Type |
|---|---|---|
| RF | 0.523 ± 0.044 | bagging |
| ExtraTrees | 0.520 ± 0.047 | bagging |
| XGBoost | 0.497 ± 0.049 | boosting |
| CatBoost | 0.477 ± 0.044 | boosting |
| LightGBM | 0.532 ± 0.052 | boosting |

Four of the five are competitive (~0.50–0.53) and CatBoost is trailing behind. On a single fingerprint, LightGBM leads, with the two bagging methods (RF, ExtraTrees) close behind and XGBoost a step below. 

### <u>Decision:</u>
Drop CatBoost. Carry RF, ExtraTrees, XGBoost, LightGBM forward as the models that will be examined further.

---

## 4. Fingerprint sweep + decorrelation

We will be using 9 fingerprints: ECFP4, ECFP6, FCFP4, FCFP6, MACCS, RDK, AVALON, TOPTOR & ATOMPAIR

<u>This section will answer two questions:</u>
1) Does any single fingerprint dominate? 
2) Which fingerprints carry _independent_ signal?


### <u>Q1. Compare AUPRC on each fingerprint seperately</u>

In [31]:
#Train a fast probe model (using XGBoost) on each fingerprint separately and compares AUPRC
ALL_FPS = ["ECFP4","ECFP6","FCFP4","FCFP6","MACCS","RDK","AVALON","ATOMPAIR","TOPTOR"]
probe = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, scale_pos_weight=spw,
                      tree_method="hist", n_jobs=-1, random_state=42, eval_metric="aucpr")

oof = {}
print(f"{'fingerprint':12s} solo AUPRC (5 grouped folds)")
for fp in ALL_FPS:
    Xf = fp_matrix(train.iloc[sub_idx], fp)
    oof[fp] = cross_val_predict(probe, Xf, y_sub, groups=groups_sub, cv=sgkf,
                                method="predict_proba", n_jobs=1)[:, 1]
    print(f"{fp:12s} {average_precision_score(y_sub, oof[fp]):.4f}")

fingerprint  solo AUPRC (5 grouped folds)
ECFP4        0.4864
ECFP6        0.4981
FCFP4        0.4522
FCFP6        0.4682
MACCS        0.3265
RDK          0.4892
AVALON       0.4791
ATOMPAIR     0.4680
TOPTOR       0.4573


**Fingerprint sweep (solo AUPRC, grouped building-block-grouped CV):** 

Top fingerprints are ECFP6 = 0.498, RDK = 0.489, ECFP4 = 0.486, and AVALON = 0.479. They all cluster within ~0.05, but there is no single dominant winner. Though MACCS does seem to trail badly at 0.327.

If no fingerprint dominates, we should ensemble several and use a multi-fingerprint approach.

### <u>Q2. Correlation matrix</u>

A good ensemble won't just combine strong models, but also ensure it combines decorrelated ones.  

This is because if two fingerprints always make the same predictions (highly correlated), averaging them is pointless as you learn nothing new from the second. But if two fingerprints are decorrelated, averaging them cancels errors and adds real information.  

So next, I want to ensemble members that are both strong as well as different from each other.

In [32]:
# decorrelation: lower correlation = complementary ensemble partners
print(pd.DataFrame(oof).corr().round(2))

          ECFP4  ECFP6  FCFP4  FCFP6  MACCS   RDK  AVALON  ATOMPAIR  TOPTOR
ECFP4      1.00   0.89   0.81   0.81   0.62  0.72    0.79      0.79    0.80
ECFP6      0.89   1.00   0.80   0.81   0.61  0.75    0.79      0.79    0.79
FCFP4      0.81   0.80   1.00   0.88   0.62  0.72    0.78      0.77    0.80
FCFP6      0.81   0.81   0.88   1.00   0.61  0.75    0.79      0.78    0.79
MACCS      0.62   0.61   0.62   0.61   1.00  0.55    0.64      0.62    0.60
RDK        0.72   0.75   0.72   0.75   0.55  1.00    0.78      0.74    0.72
AVALON     0.79   0.79   0.78   0.79   0.64  0.78    1.00      0.77    0.77
ATOMPAIR   0.79   0.79   0.77   0.78   0.62  0.74    0.77      1.00    0.78
TOPTOR     0.80   0.79   0.80   0.79   0.60  0.72    0.77      0.78    1.00


**Decorrelation matrix:** 
The Morgan family is highly correlated (ECFP4 & ECFP6 0.89, FCFP4 & FCFP6 0.88), so redundant to pair. The most *independent* signal sits in MACCS (0.55–0.64) and RDK (0.55–0.78), but MACCS seems too weak to use. 


### <u>Decision - Ensemble set selection:</u>

**Set 1: ECFP6 + RDK + ATOMPAIR (decorrelation-optimal)**
- ECFP6 strongest solo (0.498), RDK most-decorrelated strong fingerprint (0.489), ATOMPAIR distinct algorithm (0.468; is path/atom-pair-based and different from Morgan)
- Average mutual correlation ~0.76 (lowest of the three sets) 
- Circular / path-based / atom-pair
- Highest individual strength + lowest mutual correlation

**Set 2: ECFP4 + RDK + AVALON (distinct-families alternative)**
- Swaps ECFP4 with ECFP6 (avoids both Morgan-4 and Morgan-6 if Set 1 is run) and swaps AVALON for ATOMPAIR
- ECFP4 (0.486), RDK (0.489), AVALON (0.479) — all strong, RDK keeps it decorrelated
- A near-independent hedge that shares only RDK with Set 1 (backup)

**Set 3: FCFP4 + ATOMPAIR + TOPTOR (AIRCHECK published triple set)**
- The published AIRCHECK fingerprint triple for this exact WDR91 dataset — included as a literature benchmark (Wellnitz)
- Individually the weakest trio based on results here (0.452 / 0.468 / 0.457), but published precedent makes it worth including as a benchmark

---

## 5. Per-fingerprint tuning

The sets above define which fingerprints to combine. The next step is to tune each model and make the final model choice based on these results.

<u>Two design decisions:</u>. 
- Tune per fingerprint, not per set. Each ensemble averages one model per fingerprint and different fingerprints suit different tree settings. Each fingerprint will get its own `RandomizedSearchCV` (grouped CV, scored on AUPRC). The 7 unique fingerprints in our three sets (ECFP6, RDK, ATOMPAIR, ECFP4, AVALON, FCFP4, TOPTOR) are each tuned once per model.
- Each fingerprint's search uses an independent random seed, so the searches explore different candidate configurations rather than repeating the same set


### Tuning all four models per fingerprint

A `RandomizedSearchCV` per fingerprint, per model (RF, ExtraTrees, XGBoost, LightGBM), on the grouped CV. 

Across these three sets, the unique fingerprints are ECFP6, RDK, ATOMPAIR, ECFP4, AVALON, FCFP4, TOPTOR
- Since several appear in more than one set (RDK in 1 & 2, ATOMPAIR in 1 & 3), I **tuned each unique fingerprint once per model and cache the result**
- Then can assemble any set from the cached configs so the ensembles can be assembled without re-searching

In [2]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

SETS = {
    "set1": ["ECFP6", "RDK", "ATOMPAIR"],
    "set2": ["ECFP4", "RDK", "AVALON"],
    "set3": ["FCFP4", "ATOMPAIR", "TOPTOR"],
}
UNIQUE_FPS = sorted({fp for s in SETS.values() for fp in s})   # 7 unique fingerprints
print("unique fingerprints to tune per model:", UNIQUE_FPS)

unique fingerprints to tune per model: ['ATOMPAIR', 'AVALON', 'ECFP4', 'ECFP6', 'FCFP4', 'RDK', 'TOPTOR']


#### **A. Random Forest**

Starting with RandomForest, I tune each fingerprint's model individually (RandomizedSearchCV, `n_iter=20`, grouped CV, scored on AUPRC), one fingerprint at a time and cached for reuse across the three fingerprint sets.

> RandomForest is a **bagging** method (many independent trees averaged) which makes it relatively *insensitive* to hyperparameters: the averaging smooths out individual-tree variance, so defaults are often already near-optimal. But I tune it anyway for completeness and to confirm this. The search includes unbounded tree depth (`max_depth=None`), which is safe for bagging; fully-grown trees overfit individually, but averaging many of them controls the variance.

In [22]:
def tune_rf_on_fp(fp, seed=0, n_iter=20):                       
    X = fp_matrix(train.iloc[sub_idx], fp)
    dist = {"n_estimators": randint(200,600), "max_depth": [None,10,20,30],
            "max_features": ["sqrt",0.1,0.3], "min_samples_leaf": randint(1,10),
            "min_samples_split": randint(2,20)}
    s = RandomizedSearchCV(RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=42),
                           dist, n_iter=n_iter, scoring="average_precision",
                           cv=sgkf, n_jobs=1, random_state=seed, verbose=1)
    s.fit(X, y_sub, groups=groups_sub)
    print(f"{fp}: best AUPRC {s.best_score_:.4f} | {s.best_params_}")
    return s.best_params_

rf_configs = {}   # fill in with best RF params

In [30]:
rf_configs["ATOMPAIR"] = tune_rf_on_fp("ATOMPAIR", seed=0)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
ATOMPAIR: best AUPRC 0.5243 | {'max_depth': None, 'max_features': 0.1, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 451}


In [37]:
rf_configs["AVALON"] = tune_rf_on_fp("AVALON", seed=1)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
AVALON: best AUPRC 0.5267 | {'max_depth': None, 'max_features': 0.3, 'min_samples_leaf': 3, 'min_samples_split': 6, 'n_estimators': 415}


In [31]:
rf_configs["ECFP4"] = tune_rf_on_fp("ECFP4", seed=2)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
ECFP4: best AUPRC 0.5198 | {'max_depth': None, 'max_features': 0.3, 'min_samples_leaf': 3, 'min_samples_split': 6, 'n_estimators': 418}


In [32]:
rf_configs["ECFP6"] = tune_rf_on_fp("ECFP6", seed=3)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
ECFP6: best AUPRC 0.5187 | {'max_depth': None, 'max_features': 0.1, 'min_samples_leaf': 3, 'min_samples_split': 4, 'n_estimators': 356}


In [33]:
rf_configs["FCFP4"] = tune_rf_on_fp("FCFP4", seed=4)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
FCFP4: best AUPRC 0.4953 | {'max_depth': 30, 'max_features': 0.3, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 326}


In [34]:
rf_configs["RDK"] = tune_rf_on_fp("RDK", seed=5)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
RDK: best AUPRC 0.4755 | {'max_depth': 30, 'max_features': 0.3, 'min_samples_leaf': 2, 'min_samples_split': 9, 'n_estimators': 216}


In [35]:
rf_configs["TOPTOR"] = tune_rf_on_fp("TOPTOR", seed=6)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
TOPTOR: best AUPRC 0.4857 | {'max_depth': None, 'max_features': 0.1, 'min_samples_leaf': 2, 'min_samples_split': 16, 'n_estimators': 478}


In [38]:
print(f"{'fingerprint':12s} {'untuned':>8s} {'tuned':>8s} {'Δ':>7s}")
rf_untuned_oof, rf_tuned_oof = {}, {}
for fp in sorted(rf_configs.keys()):
    X = fp_matrix(train.iloc[sub_idx], fp)
    # untuned: RF defaults (n_estimators=100, all defaults) + balanced weights
    m_u = RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=42)
    p_u = cross_val_predict(m_u, X, y_sub, groups=groups_sub, cv=sgkf,
                            method="predict_proba", n_jobs=1)[:, 1]
    # tuned: cached config
    m_t = RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=42, **rf_configs[fp])
    p_t = cross_val_predict(m_t, X, y_sub, groups=groups_sub, cv=sgkf,
                            method="predict_proba", n_jobs=1)[:, 1]
    rf_untuned_oof[fp], rf_tuned_oof[fp] = p_u, p_t
    au_u, au_t = average_precision_score(y_sub, p_u), average_precision_score(y_sub, p_t)
    print(f"{fp:12s} {au_u:8.4f} {au_t:8.4f} {au_t-au_u:+7.4f}")

fingerprint   untuned    tuned       Δ
ATOMPAIR       0.5136   0.5245 +0.0109
AVALON         0.5112   0.5271 +0.0158
ECFP4          0.5176   0.5220 +0.0044
ECFP6          0.5090   0.5204 +0.0114
FCFP4          0.5057   0.4946 -0.0111
RDK            0.4611   0.4731 +0.0120
TOPTOR         0.4880   0.4857 -0.0023


In [39]:
import json

def save_configs(cfg, path):
    # convert numpy types to native python so JSON accepts them
    clean = {fp: {k: (int(v) if isinstance(v, np.integer) else float(v) if isinstance(v, np.floating) else v)
                  for k, v in c.items()} for fp, c in cfg.items()}
    json.dump(clean, open(path, "w"), indent=2)
    print(f"saved {path}: {list(cfg.keys())}")

save_configs(rf_configs,  "rf_configs_settings.json")

saved rf_configs_settings.json: ['ATOMPAIR', 'ECFP4', 'ECFP6', 'FCFP4', 'RDK', 'TOPTOR', 'AVALON']


#### **B. ExtraTrees**

ExtraTrees, the second bagging method, is tuned with the same protocol. Like RandomForest, it averages many independent trees but with an extra layer of randomness (split thresholds chosen randomly rather than optimized), so it will likely be also less sensitive to tuning.

In [33]:
def tune_et_on_fp(fp, n_iter=15, seed=0):
    X = fp_matrix(train.iloc[sub_idx], fp)
    dist = {"n_estimators": randint(300, 600), "max_depth": [20, 30, 50],
            "max_features": ["sqrt", 0.1, 0.3], "min_samples_leaf": randint(1, 10),
            "min_samples_split": randint(2, 20)}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        s = RandomizedSearchCV(
            ExtraTreesClassifier(class_weight="balanced", n_jobs=-1, random_state=8),
            dist, n_iter=n_iter, scoring="average_precision", cv=sgkf,
            n_jobs=1, random_state=seed)
        s.fit(X, y_sub, groups=groups_sub)
    print(f"{fp}: tuned AUPRC {s.best_score_:.4f} | {s.best_params_}")
    return s.best_params_

et_configs = {}   # fill in one fingerprint at a time

In [ ]:
et_configs["ATOMPAIR"] = tune_et_on_fp("ATOMPAIR", seed=0)

ATOMPAIR: tuned AUPRC 0.5362 | {'max_depth': 50, 'max_features': 0.3, 'min_samples_leaf': 3, 'min_samples_split': 2, 'n_estimators': 399}


In [ ]:
et_configs["AVALON"] = tune_et_on_fp("AVALON", seed=1)

AVALON: tuned AUPRC 0.5116 | {'max_depth': 30, 'max_features': 0.3, 'min_samples_leaf': 1, 'min_samples_split': 6, 'n_estimators': 325}


In [ ]:
et_configs["RDK"] = tune_et_on_fp("RDK", seed=2)

RDK: tuned AUPRC 0.4830 | {'max_depth': 50, 'max_features': 0.1, 'min_samples_leaf': 4, 'min_samples_split': 7, 'n_estimators': 388}


In [ ]:
et_configs["ECFP6"] = tune_et_on_fp("ECFP6", seed=3)

ECFP6: tuned AUPRC 0.5127 | {'max_depth': 30, 'max_features': 0.3, 'min_samples_leaf': 5, 'min_samples_split': 7, 'n_estimators': 484}


In [ ]:
et_configs["FCFP4"] = tune_et_on_fp("FCFP4", seed=4)

FCFP4: tuned AUPRC 0.5044 | {'max_depth': 50, 'max_features': 0.1, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 426}


In [ ]:
et_configs["ECFP4"] = tune_et_on_fp("ECFP4", seed=5)

ECFP4: tuned AUPRC 0.5267 | {'max_depth': 50, 'max_features': 0.3, 'min_samples_leaf': 4, 'min_samples_split': 3, 'n_estimators': 483}


In [ ]:
et_configs["TOPTOR"] = tune_et_on_fp("TOPTOR", seed=6)

TOPTOR: tuned AUPRC 0.4931 | {'max_depth': 50, 'max_features': 0.3, 'min_samples_leaf': 4, 'min_samples_split': 13, 'n_estimators': 429}


In [44]:
print(f"{'fingerprint':12s} {'untuned':>8s} {'tuned':>8s} {'Δ':>7s}")
et_untuned_oof, et_tuned_oof = {}, {}
for fp in sorted(et_configs.keys()):
    X = fp_matrix(train.iloc[sub_idx], fp)
    # untuned: ExtraTrees defaults + balanced weights
    m_u = ExtraTreesClassifier(class_weight="balanced", n_jobs=-1, random_state=42)
    p_u = cross_val_predict(m_u, X, y_sub, groups=groups_sub, cv=sgkf,
                            method="predict_proba", n_jobs=1)[:, 1]
    # tuned: cached config
    m_t = ExtraTreesClassifier(class_weight="balanced", n_jobs=-1, random_state=42, **et_configs[fp])
    p_t = cross_val_predict(m_t, X, y_sub, groups=groups_sub, cv=sgkf,
                            method="predict_proba", n_jobs=1)[:, 1]
    et_untuned_oof[fp], et_tuned_oof[fp] = p_u, p_t
    au_u, au_t = average_precision_score(y_sub, p_u), average_precision_score(y_sub, p_t)
    print(f"{fp:12s} {au_u:8.4f} {au_t:8.4f} {au_t-au_u:+7.4f}")

fingerprint   untuned    tuned       Δ
ATOMPAIR       0.5151   0.5390 +0.0238
AVALON         0.4919   0.5147 +0.0227
ECFP4          0.5096   0.5266 +0.0170
ECFP6          0.5023   0.5122 +0.0099
FCFP4          0.4948   0.5030 +0.0082
RDK            0.4656   0.4812 +0.0157
TOPTOR         0.4885   0.4949 +0.0064


In [45]:
save_configs(et_configs, "et_configs_settings.json")

saved et_configs_settings.json: ['ATOMPAIR', 'AVALON', 'ECFP4', 'ECFP6', 'FCFP4', 'TOPTOR', 'RDK']


#### **C. LightGBM**
LightGBM is the first **boosting** method tuned, using the same per-fingerprint protocol (RandomizedSearchCV, `n_iter=20`, grouped CV, scored on AUPRC), one fingerprint at a time and cached across the three sets.

Unlike the bagging methods (RandomForest, ExtraTrees), boosting methods are **sensitive** to hyperparameters. LightGBM adds one distinctive lever: `num_leaves`, which controls its leaf-wise tree growth (it splits wherever the gain is highest, rather than growing level-by-level).

In [ ]:
def tune_lgb_on_fp(fp, seed=0, n_iter=20):
    X = fp_matrix(train.iloc[sub_idx], fp)
    dist = {"n_estimators": randint(300,800), "num_leaves": randint(31,255),
            "learning_rate": uniform(0.01,0.1), "min_child_samples": randint(5,100),
            "subsample": uniform(0.6,0.4), "colsample_bytree": uniform(0.4,0.6),
            "reg_lambda": uniform(0,5)}
    s = RandomizedSearchCV(LGBMClassifier(is_unbalance=True, n_jobs=4, random_state=42, verbose=-1),
                           dist, n_iter=n_iter, scoring="average_precision",
                           cv=sgkf, n_jobs=1, random_state=seed)
    s.fit(X, y_sub, groups=groups_sub)
    print(f"{fp}: best AUPRC {s.best_score_:.4f} | {s.best_params_}")
    return s.best_params_

lgb_configs = {}

In [48]:
lgb_configs["ECFP6"] = tune_lgb_on_fp("ECFP6", seed=0)

ECFP6: best AUPRC 0.5754 | {'colsample_bytree': np.float64(0.6209449239043288), 'learning_rate': np.float64(0.10571551589530463), 'min_child_samples': 84, 'n_estimators': 731, 'num_leaves': 223, 'reg_lambda': np.float64(4.89309171116382), 'subsample': np.float64(0.9196634256866895)}


In [49]:
lgb_configs["RDK"] = tune_lgb_on_fp("RDK", seed=1)

RDK: best AUPRC 0.5200 | {'colsample_bytree': np.float64(0.8636434880665462), 'learning_rate': np.float64(0.025292982255402878), 'min_child_samples': 53, 'n_estimators': 578, 'num_leaves': 172, 'reg_lambda': np.float64(1.7863488000124987), 'subsample': np.float64(0.9634140603679197)}


In [50]:
lgb_configs["ATOMPAIR"] = tune_lgb_on_fp("ATOMPAIR", seed=2)

ATOMPAIR: best AUPRC 0.5528 | {'colsample_bytree': np.float64(0.42507862588918605), 'learning_rate': np.float64(0.08383997586209004), 'min_child_samples': 78, 'n_estimators': 791, 'num_leaves': 109, 'reg_lambda': np.float64(1.1469298011345201), 'subsample': np.float64(0.9516808111782131)}


In [51]:
lgb_configs["ECFP4"] = tune_lgb_on_fp("ECFP4", seed=3)

ECFP4: best AUPRC 0.5700 | {'colsample_bytree': np.float64(0.5954842948389247), 'learning_rate': np.float64(0.05451450492635321), 'min_child_samples': 38, 'n_estimators': 795, 'num_leaves': 219, 'reg_lambda': np.float64(4.858013030867838), 'subsample': np.float64(0.6922336817420296)}


In [52]:
lgb_configs["AVALON"] = tune_lgb_on_fp("AVALON", seed=4)

AVALON: best AUPRC 0.5424 | {'colsample_bytree': np.float64(0.4949340443108797), 'learning_rate': np.float64(0.06452026517032128), 'min_child_samples': 28, 'n_estimators': 414, 'num_leaves': 253, 'reg_lambda': np.float64(1.5805810839494856), 'subsample': np.float64(0.6964154411846551)}


In [53]:
lgb_configs["FCFP4"] = tune_lgb_on_fp("FCFP4", seed=5)

FCFP4: best AUPRC 0.5348 | {'colsample_bytree': np.float64(0.6714513074901853), 'learning_rate': np.float64(0.08029420771179782), 'min_child_samples': 7, 'n_estimators': 797, 'num_leaves': 250, 'reg_lambda': np.float64(2.8703674272250588), 'subsample': np.float64(0.9127659528065637)}


In [54]:
lgb_configs["TOPTOR"] = tune_lgb_on_fp("TOPTOR", seed=6)

TOPTOR: best AUPRC 0.5418 | {'colsample_bytree': np.float64(0.48897491956056593), 'learning_rate': np.float64(0.07822871306745838), 'min_child_samples': 12, 'n_estimators': 769, 'num_leaves': 209, 'reg_lambda': np.float64(1.2580707158523903), 'subsample': np.float64(0.6165502005203607)}


In [55]:
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

print(f"{'fingerprint':12s} {'untuned':>8s} {'tuned':>8s} {'Δ':>7s}")
lgb_untuned_oof, lgb_tuned_oof = {}, {}
for fp in sorted(lgb_configs.keys()):
    X = fp_matrix(train.iloc[sub_idx], fp)
    # untuned: sensible LightGBM defaults (same as the §3 model sweep)
    m_u = LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=63,
                         is_unbalance=True, n_jobs=-1, random_state=42, verbose=-1)
    p_u = cross_val_predict(m_u, X, y_sub, groups=groups_sub, cv=sgkf,
                            method="predict_proba", n_jobs=1)[:, 1]
    # tuned: cached config
    m_t = LGBMClassifier(**lgb_configs[fp], is_unbalance=True, n_jobs=-1, random_state=42, verbose=-1)
    p_t = cross_val_predict(m_t, X, y_sub, groups=groups_sub, cv=sgkf,
                            method="predict_proba", n_jobs=1)[:, 1]
    lgb_untuned_oof[fp], lgb_tuned_oof[fp] = p_u, p_t
    au_u, au_t = average_precision_score(y_sub, p_u), average_precision_score(y_sub, p_t)
    print(f"{fp:12s} {au_u:8.4f} {au_t:8.4f} {au_t-au_u:+7.4f}")

fingerprint   untuned    tuned       Δ
ATOMPAIR       0.5207   0.5526 +0.0319
AVALON         0.5101   0.5432 +0.0331
ECFP4          0.5327   0.5715 +0.0388
ECFP6          0.5370   0.5773 +0.0403
FCFP4          0.4955   0.5358 +0.0403
RDK            0.4953   0.5193 +0.0240
TOPTOR         0.4748   0.5422 +0.0673


In [56]:
save_configs(lgb_configs, "lgb_configs_settings.json")

saved lgb_configs_settings.json: ['ECFP6', 'RDK', 'ATOMPAIR', 'ECFP4', 'AVALON', 'FCFP4', 'TOPTOR']


#### **D. XGBoost**

XGBoost is the second boosting method, tuned with the same protocol (RandomizedSearchCV, `n_iter=20`, grouped CV, scored on AUPRC), one fingerprint at a time and cached for reuse across the three sets.

Like LightGBM, XGBoost is **boosting** and so responds to tuning where the bagging methods did not (learning rate, tree depth, and regularization all interact). One key difference from the bagging grids: XGBoost's `max_depth` is bounded (3–9), because boosting trees should stay shallow — each tree corrects the previous one's errors, and without averaging to control variance, deep boosting trees overfit badly. (Contrast RandomForest/ExtraTrees, where unbounded depth is safe precisely because averaging controls it)

In [ ]:
def tune_xgb_on_fp(fp, seed=0, n_iter=20):
    X = fp_matrix(train.iloc[sub_idx], fp)
    dist = {"n_estimators":     randint(200, 600),
            "max_depth":        randint(3, 10),
            "learning_rate":    uniform(0.02, 0.28),
            "subsample":        uniform(0.6, 0.4),
            "colsample_bytree": uniform(0.6, 0.4),
            "min_child_weight": randint(1, 10),
            "reg_lambda":       uniform(0, 5)}
    s = RandomizedSearchCV(
        XGBClassifier(scale_pos_weight=spw, tree_method="hist", n_jobs=-1,
                      random_state=42, eval_metric="aucpr"),
        dist, n_iter=n_iter, scoring="average_precision",
        cv=sgkf, n_jobs=1, random_state=seed, verbose=1)
    s.fit(X, y_sub, groups=groups_sub)
    print(f"{fp}: best AUPRC {s.best_score_:.4f} | {s.best_params_}")
    return s.best_params_

xgb_configs = {} 

In [12]:
xgb_configs["ATOMPAIR"] = tune_xgb_on_fp("ATOMPAIR", seed=0)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
ATOMPAIR: best AUPRC 0.5178 | {'colsample_bytree': np.float64(0.8448382890889685), 'learning_rate': np.float64(0.19274151912493195), 'max_depth': 7, 'min_child_weight': 9, 'n_estimators': 457, 'reg_lambda': np.float64(3.4881559796363244), 'subsample': np.float64(0.624090188651708)}


In [13]:
xgb_configs["AVALON"] = tune_xgb_on_fp("AVALON", seed=1)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
AVALON: best AUPRC 0.5271 | {'colsample_bytree': np.float64(0.6561547754380935), 'learning_rate': np.float64(0.07546841694376606), 'max_depth': 8, 'min_child_weight': 8, 'n_estimators': 519, 'reg_lambda': np.float64(0.4640040432036896), 'subsample': np.float64(0.8072610195767556)}


In [14]:
xgb_configs["ECFP4"] = tune_xgb_on_fp("ECFP4", seed=2)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
ECFP4: best AUPRC 0.5422 | {'colsample_bytree': np.float64(0.9047719986607997), 'learning_rate': np.float64(0.28113599803509537), 'max_depth': 9, 'min_child_weight': 6, 'n_estimators': 259, 'reg_lambda': np.float64(4.422357988489468), 'subsample': np.float64(0.8214535363308929)}


In [15]:
xgb_configs["ECFP6"] = tune_xgb_on_fp("ECFP6", seed=3)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
ECFP6: best AUPRC 0.5525 | {'colsample_bytree': np.float64(0.9030660980353199), 'learning_rate': np.float64(0.24891980324522994), 'max_depth': 7, 'min_child_weight': 2, 'n_estimators': 583, 'reg_lambda': np.float64(4.681918249302152), 'subsample': np.float64(0.9903981689891735)}


In [16]:
xgb_configs["FCFP4"] = tune_xgb_on_fp("FCFP4", seed=4)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
FCFP4: best AUPRC 0.5185 | {'colsample_bytree': np.float64(0.8475719706576375), 'learning_rate': np.float64(0.1422945617430637), 'max_depth': 9, 'min_child_weight': 8, 'n_estimators': 561, 'reg_lambda': np.float64(0.5625516704855127), 'subsample': np.float64(0.6289133039189035)}


In [17]:
xgb_configs["RDK"] = tune_xgb_on_fp("RDK", seed=5)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
RDK: best AUPRC 0.5168 | {'colsample_bytree': np.float64(0.6859916889552555), 'learning_rate': np.float64(0.06396871130406845), 'max_depth': 9, 'min_child_weight': 4, 'n_estimators': 567, 'reg_lambda': np.float64(3.2184103222299685), 'subsample': np.float64(0.6536962979885562)}


In [18]:
xgb_configs["TOPTOR"] = tune_xgb_on_fp("TOPTOR", seed=6)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
TOPTOR: best AUPRC 0.5150 | {'colsample_bytree': np.float64(0.9425152712009877), 'learning_rate': np.float64(0.1575354936680244), 'max_depth': 8, 'min_child_weight': 2, 'n_estimators': 519, 'reg_lambda': np.float64(4.022910532906876), 'subsample': np.float64(0.8455236099205465)}


In [21]:
print(f"{'fingerprint':12s} {'untuned':>8s} {'tuned':>8s} {'Δ':>8s}")
xgb_untuned_oof, xgb_tuned_oof = {}, {}
for fp in sorted(xgb_configs.keys()):
    X = fp_matrix(train.iloc[sub_idx], fp)
    # untuned baseline: sensible practitioner defaults (same as the §3 model sweep), NOT bare library defaults
    m_u = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, scale_pos_weight=spw,
                        tree_method="hist", n_jobs=-1, random_state=42, eval_metric="aucpr")
    p_u = cross_val_predict(m_u, X, y_sub, groups=groups_sub, cv=sgkf,
                            method="predict_proba", n_jobs=1)[:, 1]
    # tuned: cached config
    m_t = XGBClassifier(**xgb_configs[fp], scale_pos_weight=spw, tree_method="hist",
                        n_jobs=-1, random_state=42, eval_metric="aucpr")
    p_t = cross_val_predict(m_t, X, y_sub, groups=groups_sub, cv=sgkf,
                            method="predict_proba", n_jobs=1)[:, 1]
    xgb_untuned_oof[fp], xgb_tuned_oof[fp] = p_u, p_t
    au_u, au_t = average_precision_score(y_sub, p_u), average_precision_score(y_sub, p_t)
    print(f"{fp:12s} {au_u:8.4f} {au_t:8.4f} {au_t-au_u:+8.4f}")

fingerprint   untuned    tuned        Δ
ATOMPAIR       0.4773   0.5174  +0.0401
AVALON         0.4857   0.5278  +0.0421
ECFP4          0.4983   0.5434  +0.0451
ECFP6          0.5159   0.5538  +0.0379
FCFP4          0.4620   0.5179  +0.0560
RDK            0.5025   0.5168  +0.0144
TOPTOR         0.4616   0.5156  +0.0540


In [20]:
save_configs(xgb_configs, "xgb_configs_settings.json")

saved xgb_configs_settings.json: ['ATOMPAIR', 'AVALON', 'ECFP4', 'ECFP6', 'FCFP4', 'RDK', 'TOPTOR']


### Tuning summary: bagging vs boosting

Averaging the per-fingerprint tuning gains across all seven fingerprints:

| model | family | mean Δ AUPRC | notes |
|---|---|---|---|
| RandomForest | bagging | +0.006 | smallest gains; 2 of 7 fingerprints went slightly *negative*, defaults near-optimal |
| ExtraTrees | bagging | +0.015 | modest gains, larger than RF but still small |
| LightGBM | boosting | +0.039 | substantial, consistent gains |
| XGBoost | boosting | +0.041 | substantial, consistent gains |

**The boosting methods (LightGBM, XGBoost) gain roughly 4x more from tuning than the bagging methods (RF, ExtraTrees).**  
- This is expected as boosting builds trees sequentially with interacting hyperparameters (learning rate, depth, regularization)
- Bagging averages independent trees, which is self-regularizing, so defaults are already close to optimal and tuning mostly shuffles within noise

---

## 6. Ensemble-level model comparison

The single-fingerprint tuning above evaluate models one fingerprint at a time. The next step is to look at an **ensemble**, predictions averaged across each set's decorrelated fingerprints. How well a model's per-fingerprint predictions combine matters as much as the individual strength. So we now compare the four tuned models at the ensemble level: set-averaged AUPRC on the grouped CV.

In [146]:
tuned_oof = {"RF": rf_tuned_oof, "ExtraTrees": et_tuned_oof,
             "LightGBM": lgb_tuned_oof, "XGBoost": xgb_tuned_oof}

def set_auprc(oof, fps):
    return average_precision_score(y_sub, np.mean([oof[fp] for fp in fps], axis=0))

# all 7 combinations: 3 singles + 3 doubles + 1 triple
combos = {
    "set1":     SETS["set1"],
    "set2":     SETS["set2"],
    "set3":     SETS["set3"],
    "combined(1+2+3)": UNIQUE_FPS,
}

print(f"{'combo':10s} {'RF':>10s} {'ExtraTrees':>11s} {'LightGBM':>10s} {'XGBoost':>10s}")
for cname, fps in combos.items():
    row = "  ".join(f"{set_auprc(tuned_oof[m], fps):8.4f}" for m in ["RF","ExtraTrees","LightGBM","XGBoost"])
    print(f"{cname:10s}  {row}")

combo              RF  ExtraTrees   LightGBM    XGBoost
set1          0.5375    0.5412    0.5781    0.5717
set2          0.5418    0.5398    0.5706    0.5660
set3          0.5314    0.5401    0.5762    0.5621
combined(1+2+3)    0.5440    0.5459    0.5853    0.5826


### Model selection: LightGBM and XGBoost lead

- **Boosting (LightGBM, XGBoost) leads**: LightGBM is highest at every combination (best: combined set1+2+3, AUPRC 0.585), with XGBoost a close second.
- **Bagging (RF, ExtraTrees) trails**: both sit around ~0.54 and are essentially tied — confirming ExtraTrees captures nothing RandomForest doesn't (both average independent trees).
- **Combined set1+2+3 beats every single set** for both leading models, so the final ensembles use the full fingerprint combination.

**LightGBM and XGBoost are close — neither dominates on cross-validation**

### <u>Cross-validated performance with confidence intervals</u>

The point estimates put LightGBM marginally ahead, but a single number hides whether that gap is real or within fold-to-fold noise. 

So I report each leading model's grouped cross-validated AUROC and AUPRC with a 95% confidence interval across folds. The CI quantifies how *stable* the performance is — a wide interval would mean the score is sensitive to which building blocks land in each fold. 

In [28]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

def cv_metrics_with_ci(model_ctor, X, y, groups, n_splits=5, seed=42):
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    aurocs, auprcs = [], []
    for tr, va in sgkf.split(X, y, groups):
        m = model_ctor(); m.fit(X[tr], y[tr])
        p = m.predict_proba(X[va])[:, 1]
        aurocs.append(roc_auc_score(y[va], p))
        auprcs.append(average_precision_score(y[va], p))
    aurocs, auprcs = np.array(aurocs), np.array(auprcs)
    for name, s in [("AUROC", aurocs), ("AUPRC", auprcs)]:
        lo, hi = np.percentile(s, [2.5, 97.5])
        print(f"  {name}: {s.mean():.3f}  (95% CI {lo:.3f}–{hi:.3f})  folds={np.round(s,3).tolist()}")

groups = train["BB2"].to_numpy()[sub_idx]
X = fp_matrix(train.iloc[sub_idx], "ECFP6")

for name, ctor, cfg in [("LightGBM", lgb_ctor, lgb_configs), ("XGBoost", xgb_ctor, xgb_configs)]:
    print(f"{name} (ECFP6):")
    cv_metrics_with_ci(lambda c=ctor, g=cfg: c(g["ECFP6"]), X, y_sub, groups)
    print()

LightGBM (ECFP6):
  AUROC: 0.884  (95% CI 0.806–0.938)  folds=[0.799, 0.877, 0.901, 0.942, 0.901]
  AUPRC: 0.575  (95% CI 0.502–0.694)  folds=[0.501, 0.516, 0.589, 0.706, 0.565]

XGBoost (ECFP6):
  AUROC: 0.870  (95% CI 0.776–0.927)  folds=[0.765, 0.873, 0.896, 0.93, 0.883]
  AUPRC: 0.553  (95% CI 0.473–0.667)  folds=[0.471, 0.497, 0.576, 0.677, 0.542]



LightGBM edges XGBoost on both metrics (AUPRC 0.575 vs 0.553), but the confidence intervals overlap almost entirely on cross-validation (statistically indistinguishabl). 

This justifies carrying both forward rather than dropping one here: their in-distribution performance is effectively tied.

In [60]:
import pandas as pd
# correlation between the four models' set1+2+3 ensemble predictions (do they agree?)
ens = {m: np.mean([tuned_oof[m][fp] for fp in UNIQUE_FPS], axis=0)
       for m in ["RF","ExtraTrees","LightGBM","XGBoost"]}
print(pd.DataFrame(ens).corr().round(3))

               RF  ExtraTrees  LightGBM  XGBoost
RF          1.000       0.982     0.864    0.924
ExtraTrees  0.982       1.000     0.913    0.949
LightGBM    0.864       0.913     1.000    0.964
XGBoost     0.924       0.949     0.964    1.000


The two leading models are also **highly correlated (0.96)** 
- They rank most compounds similarly
- Hints that combining them may add little: complementarity requires *disagreement*, and these two largely agree
- Whether the small disagreements include unique hits is the question examined in the following notebook

**Decision:**  
Drop RF and ExtraTrees. Carry LightGBM and XGBoost forward as the models that will be examined further.

---

## 7. Domain-shift diagnosis

The EDA (notebook 1) showed a strong covariate shift: training (DEL) molecules are larger and more lipophilic than the E-ASMS test set, and the two sets occupy largely disjoint regions of chemical space. A model trained on DEL is therefore extrapolating on much of the test. This section asks whether explicitly correcting for that shift improves things.

<u>This is explored using three approaches:</u>
1. **Importance weighting** - reweight training compounds toward test-like ones
2. **DANN** - adversarial training for domain-invariant features
3. **Test-like selection** - evaluate models on the most test-like held-out compounds


### **<u>7.1 Importance weighting</u>**

The idea: train a **domain classifier** — a model that predicts "is this molecule from train or test?" — then use it to score each DEL training compound by how *test-like* it is. 
Those scores become **sample weights**: test-like compounds will be up-weighted, DEL-specific ones down-weighted. This will reweight DEL training molecules toward those that look test-like, so training focuses on the chemistry we'll actually later predict on.

The key diagnostic is the domain classifier's **accuracy**. If train and test are easily separable (high accuracy), then few DEL compounds look test-like and the weights collapse onto a tiny effective sample. This would mean the method cannot help, regardless of how it's later evaluated.

In [10]:
from sklearn.linear_model import LogisticRegression
from scipy.sparse import vstack

# logistic-regression domain classifier: predict "is this molecule train (0) or test (1)?" on ECFP4
n_dc = 40000
rng = np.random.default_rng(0)
tr_dc = rng.choice(len(train), n_dc, replace=False)
te_dc = rng.choice(len(test),  n_dc, replace=False)

X_dc = vstack([fp_matrix(train.iloc[tr_dc], "ECFP4"),
               fp_matrix(test.iloc[te_dc],  "ECFP4")]).tocsr()
y_dc = np.r_[np.zeros(n_dc), np.ones(n_dc)]

dom_clf = LogisticRegression(max_iter=1000).fit(X_dc, y_dc)
print("domain-classifier accuracy:", round(dom_clf.score(X_dc, y_dc), 3),
      " (near 1.0 = domains highly separable = importance weighting will struggle)")

# importance weights for the DEL training set: P(test-like) / P(DEL-like)
p_testlike = dom_clf.predict_proba(fp_matrix(train, "ECFP4"))[:, 1]
w = p_testlike / (1 - p_testlike + 1e-9)
w = np.clip(w, 0, np.percentile(w, 99))        # clip extreme weights
w = w * len(w) / w.sum()                        # normalize to mean 1
ess = w.sum()**2 / (w**2).sum()                 # effective sample size (Kish)
print(f"effective sample size: {ess:.0f} / {len(w)}  ({ess/len(w):.1%})")

domain-classifier accuracy: 1.0  (near 1.0 = domains highly separable = importance weighting will struggle)
effective sample size: 12758 / 375595  (3.4%)


**Result - importance weighting fails structurally:**
- Domain-classifier accuracy: 1.00
- DEL train and E-ASMS test are _perfectly separable_ in the fingerprint space
- A simple logistic model tells them apart with no error (confirms distinct UMAP seen from Notebook 1)
- Effective sample size: 3.4% (12,758/375,595)
- So reweighting toward test-like chemistry cuts the usable training set to a small fraction, because almost no DEL compound looks test-like

**Decision:**  
Importance weighting interpolates toward the target distribution *given overlap* between domains. Here there is essentially none so reweighting discards ~96% of the data without gaining test-relevant signal

The method collapses at the *weighting step*, before any performance metric is applied, so the negative result holds regardless of how (or on what data) is evaluated. 

There isn't enough test-like training data points

> ### Setup: identifying test-like training compounds(for methods 7.2 and 7.3)
> 
> Importance weighting (7.1) failed at the weighting step, before evaluation even mattered. The remaining two methods will produce models to evaluate and will need a definition of "test-likeness". 
> 
> I defined it by **maximum Tanimoto similarity** (binarized ECFP4): for each training compound, how similar is it to its nearest test compound? The top 20% most test-like compounds will be used to form the gap-relevant evaluation set used to fairly judge DANN (7.2) and the model ranking (7.3)

In [7]:
# === Test-like setup (used by Method 2 (DANN re-eval) and Method 3 (transfer evaluation)) ===
import numpy as np
test = pd.read_parquet(TEST_PATH) if "test" not in dir() else test
if "AlogP" in test.columns: test = test.rename(columns={"AlogP": "ALOGP"})

def fp_bin(df, idx, fp):
    sub = df.iloc[idx] if idx is not None else df
    M = np.stack(sub[fp].map(lambda s: np.array(s.split(","), dtype=np.float32)).to_numpy())
    return (M > 0).astype(np.float32)

def max_tani(T, R, chunk=3000):
    out = np.zeros(len(T)); rs = R.sum(1)
    for i in range(0, len(T), chunk):
        t = T[i:i+chunk]; inter = t @ R.T
        out[i:i+chunk] = np.nan_to_num(inter/(t.sum(1)[:,None]+rs[None,:]-inter)).max(1)
    return out

rng = np.random.default_rng(42)
test_fp = fp_bin(test, rng.choice(len(test), 20000, replace=False), "ECFP4")
train_fp = fp_bin(train, sub_idx, "ECFP4")
test_likeness = max_tani(train_fp, test_fp)
is_testlike = test_likeness >= np.percentile(test_likeness, 80)
print(f"test-like tuning compounds: {is_testlike.sum()} ({is_testlike.mean()*100:.0f}%), "
      f"{int(y_sub[is_testlike].sum())} hits ({100*y_sub[is_testlike].mean():.1f}% rate)")

test-like tuning compounds: 12116 (20%), 835 hits (6.9% rate)


### **<u>7.2 Domain-adversarial training (DANN)</u>**

DANN takes a different approach: rather than reweighting, it *learns* features in which DEL and E-ASMS are indistinguishable, via a shared feature extractor with two heads — a label head (hit/non-hit) and a domain head (train/test) fed through a gradient-reversal layer, so the extractor is trained to *fool* the domain head and discard domain-distinguishing information. The aim is a representation where the train→test gap disappears, letting the binding signal transfer.

We will train on ECFP4 with **BB2-grouped splits**, same no-leakage protocol so DANN's number is directly comparable to the other models. But a random DEL holdout measures *in-domain* fit, while DANN's actual purpose is *transfer*. Following this section's guiding principle, we therefore evaluate DANN on **both** the random DEL holdout *and* the test-like holdout — the latter being the metric that can actually see a transfer benefit, if one exists.

In [8]:
import torch, torch.nn as nn
from torch.autograd import Function
from sklearn.preprocessing import StandardScaler

class GradReverse(Function):
    @staticmethod
    def forward(ctx, x, lambd): ctx.lambd = lambd; return x.view_as(x)
    @staticmethod
    def backward(ctx, g): return -ctx.lambd * g, None
def grl(x, lambd): return GradReverse.apply(x, lambd)

class DANN(nn.Module):
    def __init__(self, d, h=128):
        super().__init__()
        self.feat   = nn.Sequential(nn.Linear(d, h), nn.ReLU(), nn.Dropout(0.3),
                                    nn.Linear(h, h), nn.ReLU())
        self.label  = nn.Linear(h, 1); self.domain = nn.Linear(h, 1)
    def forward(self, x, lambd=1.0):
        f = self.feat(x); return self.label(f), self.domain(grl(f, lambd))

In [11]:
from sklearn.model_selection import GroupShuffleSplit

# grouped fit/holdout on BB2 — no sibling leakage, same protocol as the ensembles
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
fit_i, hold_i = next(gss.split(np.zeros(len(y_train)), y_train, groups=train["BB2"].to_numpy()))

rng = np.random.default_rng(42)
src = rng.choice(fit_i, min(60000, len(fit_i)), replace=False)
Xs = fp_matrix(train.iloc[src], "ECFP4").toarray().astype(np.float32); ys = y_train[src].astype(np.float32)
Xt = fp_matrix(test.iloc[rng.choice(len(test), 60000, replace=False)], "ECFP4").toarray().astype(np.float32)
sc = StandardScaler().fit(Xs)
Xs_t, ys_t, Xt_t = torch.tensor(sc.transform(Xs)), torch.tensor(ys), torch.tensor(sc.transform(Xt))

model = DANN(Xs.shape[1]); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
bce_lab = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([(ys==0).sum()/(ys==1).sum()]))
bce_dom = nn.BCEWithLogitsLoss()
for ep in range(15):
    lambd = 2/(1+np.exp(-10*ep/15)) - 1
    perm = torch.randperm(len(Xs_t))
    for i in range(0, len(Xs_t), 512):
        b = perm[i:i+512]; td = Xt_t[torch.randint(0, len(Xt_t), (len(b),))]
        lab_logit,_ = model(Xs_t[b]); _,ds = model(Xs_t[b], lambd); _,dt = model(td, lambd)
        loss = (bce_lab(lab_logit.squeeze(), ys_t[b])
                + bce_dom(ds.squeeze(), torch.zeros(len(b))) + bce_dom(dt.squeeze(), torch.ones(len(b))))
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"epoch {ep+1}/15  loss {loss.item():.3f}  lambda {lambd:.2f}")

model.eval()
Xho = fp_matrix(train.iloc[hold_i], "ECFP4").toarray().astype(np.float32); yho = y_train[hold_i]
with torch.no_grad():
    p_dann = torch.sigmoid(model(torch.tensor(sc.transform(Xho)))[0]).squeeze().numpy()
print("DANN — grouped DEL holdout AUPRC:", round(average_precision_score(yho, p_dann), 4))

epoch 1/15  loss 1.633  lambda 0.00
epoch 2/15  loss 3.747  lambda 0.32
epoch 3/15  loss 1.428  lambda 0.58
epoch 4/15  loss 4.416  lambda 0.76
epoch 5/15  loss 8.741  lambda 0.87
epoch 6/15  loss 1.928  lambda 0.93
epoch 7/15  loss 2.148  lambda 0.96
epoch 8/15  loss 1.478  lambda 0.98
epoch 9/15  loss 1.900  lambda 0.99
epoch 10/15  loss 1.541  lambda 1.00
epoch 11/15  loss 1.863  lambda 1.00
epoch 12/15  loss 1.579  lambda 1.00
epoch 13/15  loss 1.355  lambda 1.00
epoch 14/15  loss 1.852  lambda 1.00
epoch 15/15  loss 1.342  lambda 1.00
DANN — grouped DEL holdout AUPRC: 0.4616


In [12]:
# Fair evaluation: also score DANN on the TEST-LIKE holdout (not just random DEL holdout)
# reuses p_dann, yho, hold_i, test_fp/max_tani

# test-likeness of the holdout compounds
hold_fp = (np.stack(train.iloc[hold_i]["ECFP4"].map(
    lambda s: np.array(s.split(","), dtype=np.float32)).to_numpy()) > 0).astype(np.float32)
hold_testlike = max_tani(hold_fp, test_fp)
tl_mask = hold_testlike >= np.percentile(hold_testlike, 80)

print(f"DANN — random DEL holdout AUPRC: {average_precision_score(yho, p_dann):.4f}")
print(f"DANN — test-like holdout AUPRC:  {average_precision_score(yho[tl_mask], p_dann[tl_mask]):.4f}")
print(f"(tuned ensembles ~0.58 on random DEL holdout)")

DANN — random DEL holdout AUPRC: 0.4616
DANN — test-like holdout AUPRC:  0.4526
(tuned ensembles ~0.58 on random DEL holdout)


In [24]:
# Fair single-fingerprint baseline for DANN (which used ECFP4 alone):
# tuned XGBoost and LightGBM on ECFP4, same grouped CV
print("Single-fingerprint baselines (ECFP4 alone, tuned, grouped CV AUPRC):")
print(f"  XGBoost:  {average_precision_score(y_sub, xgb_tuned_oof['ECFP4']):.4f}")
print(f"  LightGBM: {average_precision_score(y_sub, lgb_tuned_oof['ECFP4']):.4f}")
print(f"\n  DANN (ECFP4, neural):  random-holdout 0.456 | test-like 0.440")

Single-fingerprint baselines (ECFP4 alone, tuned, grouped CV AUPRC):
  XGBoost:  0.5434
  LightGBM: 0.5715

  DANN (ECFP4, neural):  random-holdout 0.456 | test-like 0.440


**Result — DANN:**
- Random DEL holdout AUPRC: 0.456
- Test-like holdout AUPRC: 0.440
- For comparison, the reference is a *single-fingerprint* model (DANN uses ECFP4 alone): tuned LightGBM and XGBoost on ECFP4 score ~0.57 and ~0.54 respectively (grouped CV). So DANN trails
> - Baselines are on the 60k grouped 5-fold CV and DANN on the BB2 grouped holdout. Slightly different splits, so comparison is approximate; but gap is wide enough that the protocol difference doesn't change the conclusion
- Training was **unstable**: loss spiked to ~20 at epoch 3 before partially settling

**Decision:**  
DANN underperforms on *both* metrics. It is **no better (slightly worse) on the test-like holdout** than on the random DEL holdout. So the failure is not an artifact of evaluating on the wrong distribution: even on the transfer-relevant metric DANN was designed to improve, it loses to the ensembles by a wide margin.

The likely cause is **concept shift**; the chemistry that distinguishes the two assays overlaps with the chemistry that drives binding. Forcing features to be "domain-blind" and domain-invariance also strips out binding-relevant signal (reflected in the unstable training). 

DANN is not adopted.

### **<u>7.3 Test-like selection</u>**

Importance weighting (7.1) collapsed structurally and DANN (7.2) failed even on the transfer metric. Both tried to reshape training to close the gap but did not work.

A lighter approach sidesteps that: keep **all** the training data (no down-weighting that discards most of it, as in 7.1) and train models normally (no competing domain objective fighting the binding objective, as in 7.2). Instead, change only *what we evaluate on*.
- Models are trained on the broad, hit-rich data as usual
- Then evaluated on the **test-like holdout**, the training compounds most similar to the test set

This asks a specific question left open by model selection: LightGBM edged XGBoost on random CV, but does that ranking *hold under transfer-aware evaluation*, or does it flip? The answer informs which model to trust for the actual test set predictions

In [71]:
from sklearn.metrics import average_precision_score
import numpy as np, warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

SET_FPS = ["ECFP6","RDK","ATOMPAIR","ECFP4","AVALON","FCFP4","TOPTOR"]

def eval_transfer_grouped(model_ctor, configs):
    """Grouped CV (no BB2 leakage); score AUPRC only on test-like compounds in each held-out fold."""
    # build out-of-fold predictions per fingerprint (grouped — siblings never split)
    oof = {}
    for fp in SET_FPS:
        Xf = fp_matrix(train.iloc[sub_idx], fp)
        oof[fp] = cross_val_predict(model_ctor(configs[fp]), Xf, y_sub,
                                    groups=groups_sub, cv=sgkf,
                                    method="predict_proba", n_jobs=1)[:, 1]
    ens = np.mean([oof[fp] for fp in SET_FPS], axis=0)
    # score on ALL vs only the TEST-LIKE compounds (both from grouped OOF — no leakage)
    all_auprc      = average_precision_score(y_sub, ens)
    testlike_auprc = average_precision_score(y_sub[is_testlike], ens[is_testlike])
    return all_auprc, testlike_auprc

print("Grouped OOF AUPRC — all compounds vs test-like subset (no BB2 leakage):")
for name, ctor, cfg in [
    ("XGBoost",  lambda c: XGBClassifier(**c, scale_pos_weight=spw, tree_method='hist', n_jobs=4, random_state=42, eval_metric='aucpr'), xgb_configs),
    ("LightGBM", lambda c: LGBMClassifier(**c, is_unbalance=True, n_jobs=4, random_state=42, verbose=-1), lgb_configs),
]:
    a, t = eval_transfer_grouped(ctor, cfg)
    print(f"  {name:9s}  all {a:.4f}  |  test-like {t:.4f}")

Grouped OOF AUPRC — all compounds vs test-like subset (no BB2 leakage):
  XGBoost    all 0.5826  |  test-like 0.5603
  LightGBM   all 0.5853  |  test-like 0.5627


**Result — test-like evaluation**

| model | all compounds | test-like subset |
|---|---|---|
| XGBoost | 0.583 | 0.560 |
| LightGBM | 0.585 | 0.563 |

Both models score lower on the test-like subset (~0.56) than on all compounds (~0.58). Supports that transfer-relevant evaluation is genuinely harder, as the chemical-space shift had predicted.

The model ranking does not change: 
- LightGBM remains marginally ahead on both standard and test-like data
- This transfer-aware evaluation does not single out a better-transferring model
- XGBoost and LightGBM are statistically indistinguishable

### **7.3b Test-like tuning**

Section 7.3 evaluated standard models on test-like compounds — it changed only the metric, not the models. A more direct intervention is to **tune the models themselves toward the test-like signal**: select hyperparameters that perform best on the transfer-relevant slice, rather than on the bulk DEL distribution.

Use the same grouped cross-validation, but each candidate is scored *only* on its most test-like validation compounds (top 20% by Tanimoto similarity to the test set). The grouping is preserved and training still uses the full, hit-rich data; only the *scoring signal* that drives hyperparameter selection is restricted to test-like compounds.

> Standard `RandomizedSearchCV` can't restrict scoring to a subset of each fold, so used an explicit grouped loop: sample candidates, run grouped CV, and score each fold only on its test-like compounds

In [5]:
import numpy as np, warnings
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import average_precision_score
from scipy.stats import randint, uniform
warnings.filterwarnings("ignore", message="X does not have valid feature names")

def tune_toward_testlike(fp, model_ctor, param_dist, n_iter=20, seed=0):
    """Grouped CV (no BB2 leakage); score each candidate ONLY on test-like validation compounds."""
    X = fp_matrix(train.iloc[sub_idx], fp)
    candidates = list(ParameterSampler(param_dist, n_iter=n_iter, random_state=seed))
    best_score, best_params = -1, None
    for params in candidates:
        fold_scores = []
        for tr_i, va_i in sgkf.split(X, y_sub, groups=groups_sub):   # GROUPED — siblings stay together
            va_tl = va_i[is_testlike[va_i]]                          # test-like compounds in this val fold
            if y_sub[va_tl].sum() < 3:                               # need a few hits to score
                continue
            m = model_ctor(params); m.fit(X[tr_i], y_sub[tr_i])
            fold_scores.append(average_precision_score(y_sub[va_tl], m.predict_proba(X[va_tl])[:, 1]))
        score = np.mean(fold_scores) if fold_scores else -1
        if score > best_score:
            best_score, best_params = score, params
    print(f"  {fp}: best test-like AUPRC {best_score:.4f}")
    return best_params

In [ ]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

SET_FPS = ["ECFP6","RDK","ATOMPAIR","ECFP4","AVALON","FCFP4","TOPTOR"]
xgb_dist = {"n_estimators": randint(200,600), "max_depth": randint(3,10),
            "learning_rate": uniform(0.02,0.28), "subsample": uniform(0.6,0.4),
            "colsample_bytree": uniform(0.6,0.4), "min_child_weight": randint(1,10),
            "reg_lambda": uniform(0,5)}
lgb_dist = {"n_estimators": randint(300,800), "num_leaves": randint(31,255),
            "learning_rate": uniform(0.01,0.1), "min_child_samples": randint(5,100),
            "subsample": uniform(0.6,0.4), "colsample_bytree": uniform(0.4,0.6),
            "reg_lambda": uniform(0,5)}
xgb_ctor = lambda p: XGBClassifier(**p, scale_pos_weight=spw, tree_method="hist", n_jobs=4, random_state=42, eval_metric="aucpr")
lgb_ctor = lambda p: LGBMClassifier(**p, is_unbalance=True, n_jobs=4, random_state=42, verbose=-1)

print("Tuning XGBoost toward test-like:") #should be quicker than bagging so just run all at once
xgb_tl_cfg = {fp: tune_toward_testlike(fp, xgb_ctor, xgb_dist, seed=i) for i, fp in enumerate(SET_FPS)}
print("Tuning LightGBM toward test-like:")
lgb_tl_cfg = {fp: tune_toward_testlike(fp, lgb_ctor, lgb_dist, seed=i) for i, fp in enumerate(SET_FPS)}

Tuning XGBoost toward test-like:
  ECFP6: best test-like AUPRC 0.5588
  RDK: best test-like AUPRC 0.5110
  ATOMPAIR: best test-like AUPRC 0.5386
  ECFP4: best test-like AUPRC 0.5506
  AVALON: best test-like AUPRC 0.5366
  FCFP4: best test-like AUPRC 0.5193
  TOPTOR: best test-like AUPRC 0.5210
Tuning LightGBM toward test-like:
  ECFP6: best test-like AUPRC 0.5976
  RDK: best test-like AUPRC 0.5168
  ATOMPAIR: best test-like AUPRC 0.5638
  ECFP4: best test-like AUPRC 0.5823
  AVALON: best test-like AUPRC 0.5560
  FCFP4: best test-like AUPRC 0.5266
  TOPTOR: best test-like AUPRC 0.5529


In [116]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import average_precision_score
SET_FPS = ["ECFP6","RDK","ATOMPAIR","ECFP4","AVALON","FCFP4","TOPTOR"]

def testlike_ensemble_auprc(ctor, cfg):
    # grouped OOF per fingerprint with the test-like-tuned configs, averaged, scored on test-like subset
    oof = {}
    for fp in SET_FPS:
        Xf = fp_matrix(train.iloc[sub_idx], fp)
        oof[fp] = cross_val_predict(ctor(cfg[fp]), Xf, y_sub, groups=groups_sub, cv=sgkf,
                                    method="predict_proba", n_jobs=1)[:, 1]
    ens = np.mean([oof[fp] for fp in SET_FPS], axis=0)
    all_a = average_precision_score(y_sub, ens)
    tl_a  = average_precision_score(y_sub[is_testlike], ens[is_testlike])
    return all_a, tl_a

xa, xt = testlike_ensemble_auprc(xgb_ctor, xgb_tl_cfg)
la, lt = testlike_ensemble_auprc(lgb_ctor, lgb_tl_cfg)
print("Test-like-TUNED ensemble (set1+2+3):")
print(f"  XGBoost:  all {xa:.4f} | test-like {xt:.4f}   (standard: 0.583 / 0.560)")
print(f"  LightGBM: all {la:.4f} | test-like {lt:.4f}   (standard: 0.585 / 0.563)")

Test-like-TUNED ensemble (set1+2+3):
  XGBoost:  all 0.5787 | test-like 0.5605   (standard: 0.583 / 0.560)
  LightGBM: all 0.5842 | test-like 0.5716   (standard: 0.585 / 0.563)


**Outcome — test-like tuning did not produce a reliable improvement.** 

The test-like-tuned ensembles were compared against the standard/normal-tuned ones, using AUPRC scored two ways: over *all* held-out compounds, and over only the *test-like* subset

| | AUPRC (all compounds) | AUPRC (test-like subset) |
|---|---|---|
| XGBoost — standard | 0.583 | 0.560 |
| XGBoost — test-like-tuned | 0.579 | 0.561 |
| LightGBM — standard | 0.585 | 0.563 |
| LightGBM — test-like-tuned | 0.584 | 0.572 |


**Decision:**
Every difference is ~0.01 AUPRC or less
- Tuning toward a proxy subset only moderately similar to the true test set (median Tanimoto ≈ 0.33) gives too weak a signal to reliably improve hyperparameter selection

### **<u>Domain-shift Conclusion</u>**
None of the three transfer-adaptation approaches reliably beat standard tuning 
- Importance weighting collapsed structurally (7.1)
- DANN failed even on the transfer metric (7.2)
- Test-like tuning produced only noise-level differences (7.3)

> **So the standard grouped-CV tuning is retained. Transfer-adaptation approaches and selecting just test-like training data will not be pursued further**

---

## 8. Confidence weighting (λ)

Each ensemble averages three per-fingerprint models. Beyond the plain average, the *spread* across those three is an uncertainty signal: a molecule all three fingerprints score highly is a more confident hit than one only a single fingerprint likes. So I re-ranked by `mean − λ·std` across the three fingerprint predictions. This rewards high-confidence agreement, demoting molecules the models disagree on. λ=0 is the plain average (the baseline to beat)

> This could also help the Hits@200 metric the Kaggle competition grades based on: with only 200 picks, we want the *most trustworthy* hits at the top.

In [117]:
LAMBDAS = [0.0, 0.25, 0.5, 0.75, 1.0]

print(f"{'model':4s} {'set':6s} {'fingerprints':25s} " + "  ".join(f"λ={l}" for l in LAMBDAS))
for mname, cache in [("LGB", lgb_tuned_oof), ("XGB", xgb_tuned_oof)]:
    for sname, fps in SETS.items():
        P = np.array([cache[fp] for fp in fps])
        row = "  ".join(f"{average_precision_score(y_sub, P.mean(0) - l*P.std(0)):.4f}" for l in LAMBDAS)
        print(f"{mname:4s} {sname:6s} {'+'.join(fps):25s} {row}")

model set    fingerprints              λ=0.0  λ=0.25  λ=0.5  λ=0.75  λ=1.0
LGB  set1   ECFP6+RDK+ATOMPAIR        0.5781  0.5806  0.5846  0.5771  0.5241
LGB  set2   ECFP4+RDK+AVALON          0.5706  0.5724  0.5747  0.5723  0.5334
LGB  set3   FCFP4+ATOMPAIR+TOPTOR     0.5762  0.5779  0.5800  0.5648  0.5118
XGB  set1   ECFP6+RDK+ATOMPAIR        0.5717  0.5728  0.5741  0.5677  0.5302
XGB  set2   ECFP4+RDK+AVALON          0.5660  0.5656  0.5642  0.5557  0.5239
XGB  set3   FCFP4+ATOMPAIR+TOPTOR     0.5621  0.5635  0.5644  0.5593  0.5352


Refined λ ∈ {0, 0.25, 0.5, 0.75, 1.0} on grouped OOF (λ>1 omitted — predictions collapse by λ=1):

| model | set | λ=0 | λ=0.25 | λ=0.5 | λ=0.75 | λ=1.0 |
|---|---|---|---|---|---|---|
| LightGBM | set1 | 0.5781 | 0.5806 | **0.5846** | 0.5771 | 0.5241 |
| LightGBM | set2 | 0.5706 | 0.5724 | **0.5747** | 0.5723 | 0.5334 |
| LightGBM | set3 | 0.5762 | 0.5779 | **0.5800** | 0.5648 | 0.5118 |
| XGBoost | set1 | 0.5717 | 0.5728 | **0.5741** | 0.5677 | 0.5302 |
| XGBoost | set2 | **0.5660** | 0.5656 | 0.5642 | 0.5557 | 0.5239 |
| XGBoost | set3 | 0.5621 | 0.5635 | **0.5644** | 0.5593 | 0.5352 |

Both boosting models show a mild interior peak at λ=0.5, then collapse at λ=1.0 (over-penalizing cross-fingerprint disagreement discards too much signal). But the gains compared to base λ=0 (third-decimal, ≤0.007) are much lower than just CV fold noise.

**<u>Decision: Not used — it works against the task</u>**  
λ rewards compounds the fingerprints agree on and demotes those they disagree on
- But given how different the test set is from training, the disagreement compounds may be precisely where the useful test-relevant signal sits
- Penalizing them risks discarding hits that don't look like typical training binders
- Since there is no major benefit/improvement, rather than use λ to filter on consensus, I will just **keep using λ=0 (the base)**

---

## 9. Final models

With the pipeline complete (per-fingerprint tuned configs), we will now build the final optimized models on the **full training set** (all 375,595 compounds, not the 60k tuning subsample).

The two strongest models, **LightGBM** and **XGBoost**, are each trained per-fingerprint across the combined **set1+2+3** ensemble (all seven fingerprints: ECFP6, RDK, ATOMPAIR, ECFP4, AVALON, FCFP4, TOPTOR) and averaged — producing the final ranked predictions for the test set. Set1+2+3 was chosen because it beat any single set for both models (Seen in §6).

In [147]:
import pandas as pd, pyarrow.parquet as pq
from scipy.stats import rankdata

ids = pq.ParquetFile(TEST_PATH).read(columns=['RandomID']).to_pandas()["RandomID"].values
SET123 = ["ECFP6","RDK","ATOMPAIR","ECFP4","AVALON","FCFP4","TOPTOR"]

def full_test_pred(ctor, cfg, fps):
    """Train each fingerprint's model on the FULL training set, predict on test, average."""
    per_fp = []
    for fp in fps:
        m = ctor(cfg[fp]); m.fit(fp_matrix(train, fp), y_train)
        per_fp.append(m.predict_proba(fp_matrix(test, fp))[:, 1])
    return np.mean(per_fp, axis=0)

# build the two strong models' full-test ensemble predictions (set1+2+3)
lgb_test = full_test_pred(lgb_ctor, lgb_configs, SET123)
xgb_test = full_test_pred(xgb_ctor, xgb_configs, SET123)
print("full-test predictions built for LightGBM and XGBoost")

full-test predictions built for LightGBM and XGBoost


### <u>Final ensemble performance with confidence intervals</u>

Before submitting, I validate the *actual* final model — the set1+2+3 ensemble (seven per-fingerprint models averaged), not any single fingerprint. 

Below is its grouped cross-validated AUROC and AUPRC with a 95% confidence interval across folds, quantifying how stable the performance is. AUPRC is the meaningful metric given the ~7.7% hit rate; the confidence interval reflects fold-to-fold variation, so a narrow interval means the ensemble's performance is consistent rather just a lucky split.

In [30]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

SET123 = ["ECFP6","RDK","ATOMPAIR","ECFP4","AVALON","FCFP4","TOPTOR"]

def ensemble_cv_ci(ctor, configs, fps, n_splits=5, seed=42):
    """CV the fingerprint-AVERAGED ensemble (set1+2+3), with 95% CI across folds."""
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    groups = train["BB2"].to_numpy()[sub_idx]
    y = y_sub
    # precompute each fingerprint's matrix once
    Xs = {fp: fp_matrix(train.iloc[sub_idx], fp) for fp in fps}
    aurocs, auprcs = [], []
    for tr, va in sgkf.split(Xs[fps[0]], y, groups):
        # train one model per fingerprint, average their held-out predictions
        fold_preds = []
        for fp in fps:
            m = ctor(configs[fp]); m.fit(Xs[fp][tr], y[tr])
            fold_preds.append(m.predict_proba(Xs[fp][va])[:, 1])
        p = np.mean(fold_preds, axis=0)   # ← the ensemble average
        aurocs.append(roc_auc_score(y[va], p))
        auprcs.append(average_precision_score(y[va], p))
    aurocs, auprcs = np.array(aurocs), np.array(auprcs)
    for nm, s in [("AUROC", aurocs), ("AUPRC", auprcs)]:
        lo, hi = np.percentile(s, [2.5, 97.5])
        print(f"  {nm}: {s.mean():.3f}  (95% CI {lo:.3f}–{hi:.3f})  folds={np.round(s,3).tolist()}")

for name, ctor, cfg in [("LightGBM", lgb_ctor, lgb_configs), ("XGBoost", xgb_ctor, xgb_configs)]:
    print(f"{name} ensemble (set1+2+3):")
    ensemble_cv_ci(ctor, cfg, SET123)
    print()

LightGBM ensemble (set1+2+3):
  AUROC: 0.898  (95% CI 0.845–0.944)  folds=[0.84, 0.885, 0.916, 0.948, 0.902]
  AUPRC: 0.584  (95% CI 0.515–0.686)  folds=[0.514, 0.526, 0.6, 0.696, 0.585]

XGBoost ensemble (set1+2+3):
  AUROC: 0.900  (95% CI 0.846–0.946)  folds=[0.842, 0.89, 0.911, 0.95, 0.907]
  AUPRC: 0.582  (95% CI 0.510–0.679)  folds=[0.509, 0.524, 0.604, 0.687, 0.585]



The ensemble improves both models over their single-fingerprint scores (LightGBM AUPRC 0.575 → 0.584; XGBoost 0.553 → 0.582), confirming that averaging across the decorrelated fingerprint sets help. 

The two models are statistically indistinguishable here — AUPRC 0.584 vs 0.582, with near-identical, overlapping confidence intervals. Cross-validation alone cannot separate them.

Whether this in-distribution tie holds on the actual E-ASMS test set will be examined. The submissions below, scored on the Kaggle leaderboard, are the real test.

### <u>Diversity-aware re-ranking</u>

A top-200 ranked purely by score can be **chemically redundant** — many near-identical molecules clustered around a few scaffolds

For drug discovery this is undesirable: screening 200 variations of the same chemotype is far less informative than 200 structurally *diverse* candidates, since diversity increases the chance of finding distinct, developable chemical series.

So we will also re-rank with a greedy **maximal-marginal-relevance (MMR)** scheme: 
- Each pick balances its model score against its similarity to compounds already selected (`score − β · max_Tanimoto_to_selected`)
- Higher β favours diversity
- Similarity is computed on **ECFP4** — the field-standard fingerprint for Tanimoto similarity and diversity selection
- ECFP4's moderate radius gives a well-calibrated similarity gradient; a larger-radius fingerprint (e.g. ECFP6) is more specific and tends to call almost every pair dissimilar, which is less useful for spreading chemical space.


> This also serves a practical purpose here: the competition's scoring includes a **redundancy/likeness penalty**, so a more diverse top-200 hedges against being penalised for structural repetition

In [148]:
# diversity re-ranking (MMR) applied to BOTH final models, at mild and moderate strengths
# beta in the 0.2-0.5 range is the field norm for diversity-vs-score selection (MMR); higher beta over-weights diversity and discards too much ranking signal
test_ecfp4 = np.stack(test["ECFP4"].map(lambda s: np.array(s.split(","), dtype=np.float32)).to_numpy())

div_preds = {}   # (model_name, beta) -> reranked scores
for mname, base_pred in [("lightgbm", lgb_test), ("xgboost", xgb_test)]:
    for beta in [0.2, 0.4]:
        order = diverse_rerank(base_pred, test_ecfp4, top_pool=2000, beta=beta)
        div_score = np.zeros(len(base_pred))
        for rank, idx in enumerate(order):
            div_score[idx] = len(order) - rank
        div_preds[(mname, beta)] = div_score
        print(f"{mname} beta={beta}: diversity-reranked predictions built")

lightgbm beta=0.2: diversity-reranked predictions built
lightgbm beta=0.4: diversity-reranked predictions built
xgboost beta=0.2: diversity-reranked predictions built
xgboost beta=0.4: diversity-reranked predictions built


### <u>Building submission files</u>

For the Kaggle competition, I will be submitting the strongest single models as well as the diversity-hedged variants:

In [149]:
def write_submission(scores, filename):   # for the Kaggle submission CSVs
    pd.DataFrame({"RandomID": ids, "DELLabel": scores}).to_csv(filename, index=False)
    print(f"saved {filename}")

# primary single-model submissions
write_submission(lgb_test, "submission_lightgbm.csv")
write_submission(xgb_test, "submission_xgboost.csv")

# diversity-hedged submissions — both models, at mild (0.2) and moderate (0.4) diversity
for (mname, beta), scores in div_preds.items():
    write_submission(scores, f"submission_{mname}_beta{beta}.csv")

saved submission_lightgbm.csv
saved submission_xgboost.csv
saved submission_lightgbm_beta0.2.csv
saved submission_lightgbm_beta0.4.csv
saved submission_xgboost_beta0.2.csv
saved submission_xgboost_beta0.4.csv


### <u>Leaderboard results in Kaggle</u>
After submission of the CSVs, the public score results were:

| Submission | Hits@200 |
|---|---|
| LightGBM | 9 |
| LightGBM (β=0.2) | 9 |
| LightGBM (β=0.4) | 7 |
| XGBoost | 4 |
| XGBoost (β=0.2) | 5 |
| XGBoost (β=0.4) | 3 |

**LightGBM was the stronger model** 
— 9 hits in the top-200 versus XGBoost's 4, despite the two being statistically indistinguishable on cross-validation (ensemble AUPRC 0.584 vs 0.582, with heavily overlapping confidence intervals) 
- This is the cross-validation-vs-test gap that the domain-shift analysis anticipated: two models tied in-distribution diverge by more than 2× on transfer — comparable cross-validated performance, even with confidence intervals, does not predict which model generalizes to the E-ASMS test set

**Diversity hedging behaved as intended** 
- Mild diversity (β=0.2) preserved all 9 LightGBM hits and should be spreading the top-200 across more chemical space 
- Moderate diversity (β=0.4) traded 2 hits for further spread

**The final single model is the tuned LightGBM ensemble (set1+2+3)** that will be examined further and combined with different models in the following notebooks

---

## 10. Conclusion

This notebook built a ranking pipeline through multiple steps:

**Model selection**:  
- A five-model survey (RF, ExtraTrees, XGBoost, CatBoost, LightGBM) narrowed to two strong boosting learners — **LightGBM and XGBoost**.  
- At the ensemble level, the combined **set1+2+3** fingerprint ensemble beat any single set for both models.

**Tuning and refinements**  
- Per-fingerprint hyperparameter tuning helped the boosting models 
- Confidence weighting (λ) was tested and not adopted — its gains were within noise and may work against this task (where the valuable hits are the unusual compounds that fingerprints disagree on)

**Domain shift**  
- The train→test gap was examined using three methods
- Importance weighting collapsed structurally
- DANN failed even on a transfer-aware metric
- Test-like tuning produced only noise-level differences
- No adaptation method beat standard tuning — the shift is real but seems best handled by a strong, standard-tuned ranker

---
---

## Next Steps

LightGBM alone is the strongest single approach. But two questions remain open:  

1. **Does a fundamentally different model architecture help?** Every model here is a tree ensemble. Notebook 3 will test TabPFN, a transformer-based foundation model for tabular data (might capture signal the trees miss).

2. **Does combining models help?** LightGBM and XGBoost are strong but highly correlated (0.96), which already hints that combination of them may add little. Notebook 4 will test this through stacking, rank-fusion, and adding diverse models (including TabPFN) — to see whether any combination beats the strong single LightGBM, or whether, for this task, the best single model is hard to improve on.


<u>Save files for notebook 4:</u>

In [ ]:
# Run in this notebook 2, where lgb_tuned_oof/xgb_tuned_oof exist
import numpy as np
np.save("lgb_oof.npy", np.mean([lgb_tuned_oof[fp] for fp in SET123], axis=0))
np.save("xgb_oof.npy", np.mean([xgb_tuned_oof[fp] for fp in SET123], axis=0))
np.save("y_sub.npy", y_sub) #hit/non-hit labels (0/1) for the 60k subset
print("saved OOF arrays for NB4")

saved OOF arrays for NB4


In [ ]:
#RF (set1+2+3) for notebook 4
rf_test = full_test_pred(rf_ctor, rf_configs, SET123)
np.save("rf_test_pred.npy", rf_test)

In [23]:
#ExtraTrees (set1+2+3) for notebook 4
et_test = full_test_pred(et_ctor, et_configs, SET123)
np.save("et_test_pred.npy", et_test)